# MLP Classification with Unitary Evolution

This notebook contains an example of how we can use Hamiltonian Learning to replicate the effect of a Neural Network with the unitary evolution of a quantum state

## Imports

In [3]:
import numpy as np
import torch
from torchvision import transforms, datasets
import tensorkrowch as tk

import jax
import jax.numpy as jnp

import torch.nn as nn
import torch.nn.functional as F

import pandas as pd
import glob
import yaml
import matplotlib.pyplot as plt
import torchtt as tntt 

import sys
import os

sys.path.append('../src/')

from run_inference import run_inference
from plots import bar_plot_strings_comparison, plot_training_loss
from physics import paulis, kron_n, rk4_step, build_hamiltonian, schrodinger_rhs, evolve_state, time_evolution, physics_computation
from physics import OperatorClass, prepare_initial_state, initial_state_from_input
from NeuralNetworks import MPS_MLP

## Utils specific to this notebook

In [4]:
def load_config(config_path):
    '''Load configuration from YAML file'''
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

def create_filename_core(config, N):
    chi_nn = config['bond_dimension_learning']
    kind = config['learning_mode']
    chi_mpo = config['MAX_MPO_CHI']
    mpo_on = config['MPO_ON']
    
    filename_core = f"L{N}_chi_nn-{chi_nn}_kind-{kind}_Chimpo-{chi_mpo}_MPO-{mpo_on}_class-7"

    return filename_core

def generate_bitstring_list(nqubits):
    '''Create list containing all possible bitstrings of the N-qubit chain'''
    bitstrings = []
    decimal_bitstrings = range(0, 2**nqubits)
    int_bitstrings = [bin(i)[2:].zfill(nqubits) for i in decimal_bitstrings]
    bitstrings =  [str(bit) for bit in int_bitstrings]

    return bitstrings

def min_power_of_2(n):
    if n <= 0:
        raise ValueError("Number must be positive")
    
    # If n is already a power of 2, return its exponent
    if (n & (n - 1)) == 0:
        return n.bit_length() - 1
    
    # Otherwise, return the bit length
    return n.bit_length()

def create_parameter_dict(params, OPS_LIST, L, CONFIG):
    """
    Create parameter dictionary from model output.
    FIXED: Better handling of parameter shapes.
    """
    predicted_params = {}
    
    # Ensure params is 1D
    if params.ndim > 1:
        params = params.squeeze()
    
    if params.ndim > 1:
        raise ValueError(f"params should be 1D after squeeze, got shape {params.shape}")
    
    idx = 0
    
    # 1. Hamiltonian parameters
    n_hamiltonian = len(OPS_LIST)
    predicted_params['theta'] = params[idx:idx + n_hamiltonian]
    idx += n_hamiltonian
    
    #Not used
    for field in ['x', 'y', 'z']:
        key = f'{field}_fields'
        if CONFIG.get(key, False):
            predicted_params[f'rot_{field}'] = params[idx:idx + L]
            idx += L
    
    # Verify we used all parameters
    if idx != params.shape[0]:
        raise ValueError(
            f"Parameter count mismatch. Used {idx} parameters, "
            f"but model output has {params.shape[0]}"
        )
    
    return predicted_params

## Training

In [5]:
def train_model(model, n_epochs, input_data, psi0, OPS_LIST, L, CONFIG, 
                        t_grid_fine, learning_rate, counts_shots, print_every=50):
    """
    Improved training function with better convergence.
    """
    
    loss_history = []
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5
    )
    
    for epoch_i in range(n_epochs):

        optimizer.zero_grad()
        
        # Forward pass
        output_params = model(input_data)
        predicted_params = create_parameter_dict(output_params, OPS_LIST, L, CONFIG)
        
        # Physics computation
        psi_t = physics_computation(predicted_params, psi0, OPS_LIST, L, t_grid_fine)
        
        # Compute loss
        loss = nll(psi_t, counts_shots)
        
        # Add regularization to prevent parameter explosion
        if CONFIG.get('lambda_reg', 0) > 0:
            param_norm = sum(p.norm() for p in predicted_params.values() if isinstance(p, torch.Tensor))
            loss = loss + CONFIG['lambda_reg'] * param_norm
        
        # Backward pass
        loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Optimization step
        optimizer.step()
        
        # Record loss
        loss_val = loss.item()
        loss_history.append(loss_val)
        
        # Learning rate scheduling
        scheduler.step(loss_val)
        
        # Print progress
        if epoch_i % print_every == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch_i:4d} | Loss: {loss_val:.6f} | LR: {current_lr:.2e}")
    
    return model, predicted_params, psi_t, loss_history